# AIRPTCoach EXP02 — LiteRT Export & Verification

이 노트북은 기존 `AIRPTCoach_AIHub_AutoPipeline.ipynb`에서 EXP02 모델의
모바일 export/검증에 필요한 부분만 분리한 최소 재현 파일입니다.

## 목적
- EXP02 `best.pt` 존재 확인
- Ultralytics 8.4.176 기준 LiteRT export
- 최종 `.tflite` 파일 위치/크기 확인
- Ultralytics wrapper를 통한 test split 검증
- raw TFLite/LiteRT tensor shape/dtype 확인
- Android 통합에 필요한 "확인된 사실"과 "아직 미확인인 계약"을 분리

## 중요
이 파일은 Android 전처리/후처리 계약을 임의로 추측하지 않습니다.

현재 원본 실험에서 직접 확인된 항목:
- class 0 = `SQUAT`
- class 1 = `SHOULDER_PRESS`
- input = FLOAT32 `[1, 3, 640, 640]` (BCHW)
- output = FLOAT32 `[1, 6, 8400]`
- export = `format="litert", imgsz=640, batch=1, device="cpu"`

아직 원본 실험 코드만으로 직접 확정되지 않은 항목:
- RGB/BGR 처리 세부사항
- resize/letterbox/padding 구현 세부사항
- normalization 세부사항
- output 6개 channel의 직접 의미
- xywh/xyxy 및 좌표 단위
- confidence/class score 결합 방식
- NMS 위치/세부 구현

위 미확인 항목은 Ultralytics 8.4.176의 실제 LiteRT inference pipeline을 기준으로 확인한 후
Android `YoloPreprocessor`, `YoloOutputDecoder`, `YoloPostProcessor`에 반영해야 합니다.


In [ ]:
from pathlib import Path
import shutil
import sys

# Google Drive mount가 필요한 경우 먼저 실행:
# from google.colab import drive
# drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/AIRPTCoach_ML")
MODEL_DIR = DRIVE_ROOT / "05_models"

EXP02_PT = MODEL_DIR / "AIRPTCoach_EXP02_YOLO26n_best.pt"
FINAL_TFLITE = MODEL_DIR / "airptcoach_yolo26n_exp02_fp32.tflite"

# 원본 EXP02 test dataset yaml.
# 실제 검증을 다시 실행하려면 이 경로가 현재 런타임에 존재해야 합니다.
EXP02_YAML = Path("/content/airptcoach_baseline/yolo_dataset_v2/data.yaml")

print("Python:", sys.version.split()[0])
print("EXP02 PT:", EXP02_PT)
print("FINAL TFLite:", FINAL_TFLITE)
print("Dataset YAML:", EXP02_YAML)


## 1. 파일 존재 확인

원본 대형 노트북의 export 단계는 아래 EXP02 weight를 사용했습니다.

`/content/drive/MyDrive/AIRPTCoach_ML/05_models/AIRPTCoach_EXP02_YOLO26n_best.pt`


In [ ]:
for path in [EXP02_PT, FINAL_TFLITE, EXP02_YAML]:
    print(f"{path.name}: {'FOUND' if path.exists() else 'MISSING'}")
    if path.exists() and path.is_file():
        print(f"  size = {path.stat().st_size / 1024 / 1024:.2f} MB")


## 2. EXP02 → LiteRT export

원본 실험에서 사용한 export 명령을 그대로 보존합니다.

원본 실행 환경 기록:
- Ultralytics 8.4.176
- Python 3.13.15
- `litert_torch` 0.9.4
- `ai-edge-litert` 2.2.0
- input `(1, 3, 640, 640)` BCHW
- output `(1, 6, 8400)`

실수로 기존 모델을 덮어쓰지 않도록 기본값은 `RUN_EXPORT = False` 입니다.
다시 export하려는 경우에만 명시적으로 `True`로 바꾸세요.


In [ ]:
RUN_EXPORT = False

if RUN_EXPORT:
    if not EXP02_PT.exists():
        raise FileNotFoundError(f"EXP02 weight not found: {EXP02_PT}")

    from ultralytics import YOLO

    model = YOLO(str(EXP02_PT))
    exported_model = model.export(
        format="litert",
        imgsz=640,
        batch=1,
        device="cpu",
    )

    print("Export result:", exported_model)
else:
    print("Export skipped. Set RUN_EXPORT=True only when a new export is intentionally required.")


## 3. 생성된 LiteRT 파일을 canonical 이름으로 정리

원본 실험에서는 Ultralytics가 생성한
`AIRPTCoach_EXP02_YOLO26n_best.tflite`를
`airptcoach_yolo26n_exp02_fp32.tflite`로 복사했습니다.

이미 canonical 파일이 존재한다면 다시 복사할 필요가 없습니다.


In [ ]:
RUN_CANONICAL_COPY = False

export_candidates = sorted(MODEL_DIR.glob("AIRPTCoach_EXP02_YOLO26n_best*.tflite"))

print("Export candidate count:", len(export_candidates))
for candidate in export_candidates:
    print(" -", candidate.name, f"{candidate.stat().st_size / 1024 / 1024:.2f} MB")

if RUN_CANONICAL_COPY:
    if not export_candidates:
        raise FileNotFoundError("No exported EXP02 .tflite candidate found.")
    source = export_candidates[0]
    shutil.copy2(source, FINAL_TFLITE)
    print("Copied:", source, "->", FINAL_TFLITE)
else:
    print("Copy skipped. Existing canonical model is preserved.")


## 4. Ultralytics wrapper를 통한 LiteRT test 검증

원본 실험에서는 raw tensor를 직접 decode하지 않고,
`YOLO(tflite_path).val(...)`로 LiteRT 모델을 검증했습니다.

따라서 이 셀은 "모델이 Ultralytics 8.4.176 pipeline에서 정상 동작했는지"를 검증하는 용도입니다.

원본 노트북에 저장된 역사적 LiteRT test 결과:
- images: 73
- backgrounds: 14
- instances: 59
- Precision: 0.9339
- Recall: 0.8913
- mAP50: 0.9383
- mAP50-95: 0.8652

클래스별 저장 결과:
- SQUAT: P 1.000 / R 0.783 / mAP50 0.886 / mAP50-95 0.829
- SHOULDER_PRESS: P 0.868 / R 1.000 / mAP50 0.991 / mAP50-95 0.902

위 값은 "원본 노트북에 저장된 과거 실행 결과"이며,
이 정리 노트북을 새로 실행해 재측정한 값이 아닙니다.


In [ ]:
RUN_LITERT_VALIDATION = False

if RUN_LITERT_VALIDATION:
    if not FINAL_TFLITE.exists():
        raise FileNotFoundError(f"LiteRT model not found: {FINAL_TFLITE}")
    if not EXP02_YAML.exists():
        raise FileNotFoundError(f"Dataset YAML not found: {EXP02_YAML}")

    from ultralytics import YOLO

    litert_model = YOLO(str(FINAL_TFLITE))
    metrics = litert_model.val(
        data=str(EXP02_YAML),
        split="test",
        imgsz=640,
    )

    print("=" * 60)
    print("EXP02 LiteRT TEST RESULTS")
    print("=" * 60)
    print(f"Precision : {float(metrics.box.mp):.4f}")
    print(f"Recall    : {float(metrics.box.mr):.4f}")
    print(f"mAP50     : {float(metrics.box.map50):.4f}")
    print(f"mAP50-95  : {float(metrics.box.map):.4f}")
else:
    print("Validation skipped. Set RUN_LITERT_VALIDATION=True only when dataset/test environment is ready.")


## 5. Raw tensor metadata 확인

이 셀은 전처리/후처리를 추측하지 않고 모델의 실제 input/output tensor metadata만 확인합니다.

원본 실험에서 확인된 값:
- input name: `serving_default_args_0`
- input shape: `[1, 3, 640, 640]`
- input dtype: `float32`
- input quantization: `(0.0, 0)`
- output name: `serving_default_output_0_output`
- output shape: `[1, 6, 8400]`
- output dtype: `float32`
- output quantization: `(0.0, 0)`


In [ ]:
if not FINAL_TFLITE.exists():
    print("TFLite model not found; tensor inspection skipped.")
else:
    # TensorFlow가 설치되어 있으면 기존 원본과 같은 방식으로 검사합니다.
    # 새 환경에서는 ai_edge_litert를 사용할 수 있지만,
    # 여기서는 원본 실험과의 비교 가능성을 위해 TensorFlow fallback도 허용합니다.
    try:
        from ai_edge_litert.interpreter import Interpreter
        backend = "ai_edge_litert"
    except Exception:
        import tensorflow as tf
        Interpreter = tf.lite.Interpreter
        backend = "tensorflow.lite"

    interpreter = Interpreter(model_path=str(FINAL_TFLITE))
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    print("Backend:", backend)

    print("\nINPUT")
    for item in input_details:
        print(" name:", item["name"])
        print(" shape:", item["shape"])
        print(" dtype:", item["dtype"])
        print(" quantization:", item["quantization"])

    print("\nOUTPUT")
    for item in output_details:
        print(" name:", item["name"])
        print(" shape:", item["shape"])
        print(" dtype:", item["dtype"])
        print(" quantization:", item["quantization"])


## 6. Android 연동용 계약 요약

### 현재 VERIFIED 가능한 사실
- 모델 파일: `airptcoach_yolo26n_exp02_fp32.tflite`
- 모델 계열: Ultralytics YOLO26n detection
- classes:
  - `0 = SQUAT`
  - `1 = SHOULDER_PRESS`
- export:
  - `format="litert"`
  - `imgsz=640`
  - `batch=1`
  - `device="cpu"`
- input:
  - FLOAT32
  - `[1, 3, 640, 640]`
  - BCHW
- output:
  - FLOAT32
  - `[1, 6, 8400]`
- 원본 검증:
  - `YOLO(FINAL_TFLITE).val(...)`
  - Ultralytics 8.4.176 pipeline 사용

### 아직 NOT_VERIFIED
다음 항목을 tensor shape만으로 추측하지 않습니다.
- RGB/BGR
- exact resize/letterbox implementation
- padding value
- normalization
- output channel semantics
- bbox raw format
- coordinate scale
- confidence/class score calculation
- NMS location/implementation
- Android threshold

### 다음 단계
1. Ultralytics 8.4.176 LiteRT predictor의 실제 preprocessing/postprocessing 구현 확인
2. Android `YoloPreprocessor` 구현
3. Android `YoloOutputDecoder` 구현
4. Android `YoloPostProcessor` 구현
5. 같은 입력에 대해 Python/Ultralytics와 Android 결과를 비교하는 golden test 작성
6. 일치가 확인된 뒤에만 Shadow Mode actual prediction 활성화
7. 그 이후 실기기 FRONT/SIDE 비교
8. 충분한 검증 후에만 YOLO Primary 전환 검토

이 파일은 학습 전체 파이프라인을 재현하기 위한 노트북이 아니라
"EXP02 모바일 export와 Android inference 계약 검증"을 위한 최소 문서입니다.
